In [2]:
import pandas as pd
import os
import json
from utils import (
    write_parquet,
    get_df,
    compare_dataframes,
    get_size,
    filter_sample_df,
    get_pre_suffix
)

dataset_id = "nhagar/fineweb_urls"
dataset_config = ""
configs_file = "config.json"
config_dict = json.load(open(configs_file, "r"))[dataset_id]

In [3]:
df = get_df(config_dict['configs'][0]['path'])
print(df.head())

                                                 url                   domain
0  http://0ncemorewithfeeling.blogspot.com/2010/1...             blogspot.com
1  http://1001plus.blogspot.com/2011/11/very-shor...             blogspot.com
2  http://101bestandbrightest.com/companies/aviso...  101bestandbrightest.com
3  http://101bestandbrightest.com/companies/field...  101bestandbrightest.com
4    http://10pointsmanagement.com/the-wizard-of-oz/   10pointsmanagement.com


In [4]:

sampled_df, string_columns = filter_sample_df(df)
# sampled_df = sampled_df.map(lambda x: x.replace('"', '') if isinstance(x, str) else x)
# sampled_df = sampled_df[['product_name', 'product_hashisy', 'product_smiles']]
# sampled_df = sampled_df.drop(columns=['pains_filter_match_name', 'lhasa_scoring_history', 'lhasa_reaction_warnings', 'pains_filter_match_name'])
string_columns = sampled_df.select_dtypes(include=['object', 'string']).columns


In [5]:
results = get_pre_suffix(sampled_df, string_columns)

Comparing 2 string columns for prefix and suffix matches...
Filter threshold set to 0.90


In [6]:
results

[{'col1': 'url',
  'col2': 'domain',
  'containment_ratio': np.float64(1.0),
  'infix_match': np.float64(1.0)}]

In [11]:
# statistic len of domain
sampled_df['domain_len'] = sampled_df['domain'].apply(lambda x: len(x))
# min and max length of domain
print(f"Min domain length: {sampled_df['domain_len'].min()}")
print(f"Max domain length: {sampled_df['domain_len'].max()}")
print(f"Mean domain length: {sampled_df['domain_len'].mean()}")

df['domain_offset'] = df.apply(lambda row: row['url'].find(row['domain']) if row['url'].find(row['domain']) > 0 else 0, axis=1)
# min and max offset of domain in url
print(f"Min domain offset: {df['domain_offset'].min()}")
print(f"Max domain offset: {df['domain_offset'].max()}")
print(f"Mean domain offset: {df['domain_offset'].mean()}")


Min domain length: 5
Max domain length: 42
Mean domain length: 14.7158
Min domain offset: 0
Max domain offset: 84
Mean domain offset: 12.637715


In [6]:
config_dict = json.load(open(configs_file, "r"))
dataset_ids = config_dict.keys()
for dataset_id in dataset_ids:
    print(f"\n\nProcessing dataset: {dataset_id}")
    config = config_dict[dataset_id]
    df = get_df(config['configs'][0]['path'])
    sampled_df, string_columns = filter_sample_df(df)
    results = get_pre_suffix(sampled_df, string_columns)
    print(f"\nResults: {results}")
    with open(f"results/{dataset_id.replace("/", "_")}_results.json", "w") as f:
        json.dump(results, f, indent=4)



Processing dataset: wikimedia/wikipedia
Comparing 4 string columns for prefix and suffix matches...
Filter threshold set to 0.90
Processing (12/12)
Results: [{'col1': 'text', 'col2': 'title', 'containment_ratio': np.float64(0.9094426287601287)}]


Processing dataset: HuggingFaceFW/fineweb
Comparing 7 string columns for prefix and suffix matches...
Filter threshold set to 0.90
Processing (42/42)
Results: [{'col1': 'file_path', 'col2': 'dump', 'containment_ratio': np.float64(1.0), 'infix_match': np.float64(1.0)}]


Processing dataset: Metanova/SAVI-2020
Comparing 38 string columns for prefix and suffix matches...
Filter threshold set to 0.90
Processing (1406/1406)
Results: [{'col1': 'product_name', 'col2': 'product_hashisy', 'containment_ratio': np.float64(0.9444444444444445), 'prefix_match': np.float64(1.0)}, {'col1': 'product_name', 'col2': 'product_stereo_tauto_hash', 'containment_ratio': np.float64(0.9134222222222222), 'prefix_match': np.float64(0.9712)}, {'col1': 'product_hashisy'

In [6]:
config_dict = json.load(open(configs_file, "r"))
dataset_ids = ["vCache/SemBenchmarkLmArena", "vCache/SemBenchmarkClassification"]
for dataset_id in dataset_ids:
    print(f"\n\nProcessing dataset: {dataset_id}")
    config = config_dict[dataset_id]
    df = get_df(config['configs'][0]['path'])
    sampled_df, string_columns = filter_sample_df(df)
    print(f"String columns: {string_columns}")
    string_columns = string_columns[[not col.startswith("emb") for col in string_columns]]
    results = get_pre_suffix(sampled_df, string_columns)
    print(f"\nResults: {results}")
    with open(f"results/{dataset_id.replace("/", "_")}_results.json", "w") as f:
        json.dump(results, f, indent=4)



Processing dataset: vCache/SemBenchmarkLmArena


String columns: Index(['task', 'dataset_name', 'output_format', 'text',
       'emb_text-embedding-3-large', 'emb_text-embedding-3-small',
       'response_gpt-4o-mini', 'response_gpt-4.1-nano', 'emb_gte',
       'emb_gte_ft', 'emb_e5_large_v2', 'emb_e5_large_v2_ft'],
      dtype='object')
Comparing 6 string columns for prefix and suffix matches...
Filter threshold set to 0.90
Processing (30/30)
Results: [{'col1': 'task', 'col2': 'text', 'equal_match': np.float64(1.0)}, {'col1': 'text', 'col2': 'task', 'equal_match': np.float64(1.0)}]


Processing dataset: vCache/SemBenchmarkClassification
String columns: Index(['dataset_name', 'task', 'output_format', 'text', 'emb_gte',
       'emb_gte_ft', 'emb_e5_mistral_7b', 'emb_e5_large_v2',
       'emb_e5_large_v2_ft', 'response_llama_3_8b', 'response_llama_3_70b'],
      dtype='object')
Comparing 6 string columns for prefix and suffix matches...
Filter threshold set to 0.90
Processing (30/30)
Results: [{'col1': 'output_format', 'col2': 'respons